# markdown



In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)


def enrich_paysim(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()
    n = len(df)

    fraud_mask  = df["isFraud"] == 1
    legit_mask  = df["isFraud"] == 0
    fraud_count = fraud_mask.sum()
    legit_count = legit_mask.sum()

    print(f"Total : {n:,}  |  Fraud : {fraud_count:,}  |  Legit : {legit_count:,}\n")

    # ── 1. transaction_hour ──────────────────────────────────────
    df["transaction_hour"] = (df["step"] % 24).astype("int8")
    print("[1/13] transaction_hour done")

    # ── 2. transaction_day_of_week ───────────────────────────────
    df["transaction_day_of_week"] = ((df["step"] // 24) % 7).astype("int8")
    print("[2/13] transaction_day_of_week done")

    # ── 3. time_of_day ───────────────────────────────────────────
    h = df["transaction_hour"]
    conditions = [
        (h >= 1)  & (h <= 5),
        (h >= 6)  & (h <= 11),
        (h >= 12) & (h <= 17),
        (h >= 18) & (h <= 23),
        (h == 0),
    ]
    choices = ["late_night", "morning", "afternoon", "evening", "midnight"]
    df["time_of_day"] = pd.Categorical(
        np.select(conditions, choices, default="morning"),
        categories=choices
    )
    print("[3/13] time_of_day done")

    # ── 4. customer_age ──────────────────────────────────────────
    age = np.empty(n, dtype="float32")
    age[legit_mask] = np.random.normal(38, 12, legit_count)
    age[fraud_mask] = np.random.normal(52, 15, fraud_count)
    df["customer_age"] = np.clip(age, 18, 75).astype("int8")
    print("[4/13] customer_age done")

    # ── 5. customer_gender ───────────────────────────────────────
    opts = ["Male", "Female", "Other"]
    g = np.empty(n, dtype=object)
    g[legit_mask] = np.random.choice(opts, legit_count, p=[0.52, 0.45, 0.03])
    g[fraud_mask] = np.random.choice(opts, fraud_count, p=[0.58, 0.39, 0.03])
    df["customer_gender"] = pd.Categorical(g, categories=opts)
    print("[5/13] customer_gender done")

    # ── 6. account_age_days ──────────────────────────────────────
    aad = np.empty(n, dtype="float32")
    aad[legit_mask] = np.random.normal(1200, 800, legit_count)
    aad[fraud_mask] = np.random.normal(45,   30,  fraud_count)
    df["account_age_days"] = np.clip(aad, 1, 3650).astype("int16")
    print("[6/13] account_age_days done")

    # ── 7. is_new_account ────────────────────────────────────────
    df["is_new_account"] = (df["account_age_days"] < 90).astype("int8")
    print("[7/13] is_new_account done")

    # ── 8. device_type ───────────────────────────────────────────
    dev_opts = ["Mobile", "Desktop", "Tablet", "ATM", "POS"]
    dev = np.empty(n, dtype=object)
    dev[legit_mask] = np.random.choice(dev_opts, legit_count, p=[0.45, 0.30, 0.10, 0.10, 0.05])
    dev[fraud_mask] = np.random.choice(dev_opts, fraud_count, p=[0.55, 0.12, 0.03, 0.25, 0.05])
    df["device_type"] = pd.Categorical(dev, categories=dev_opts)
    print("[8/13] device_type done")

    # ── 9. channel (correlated with device_type) ─────────────────
    ch = np.empty(n, dtype=object)
    ch[df["device_type"] == "ATM"] = "ATM"
    ch[df["device_type"] == "POS"] = "POS"

    for device, probs in [("Mobile", [0.85, 0.15]),
                           ("Desktop",[0.20, 0.80]),
                           ("Tablet", [0.55, 0.45])]:
        idx = np.where(df["device_type"] == device)[0]
        ch[idx] = np.random.choice(["App", "Web"], len(idx), p=probs)

    df["channel"] = pd.Categorical(ch, categories=["App", "Web", "ATM", "POS"])
    print("[9/13] channel done")

    # ── 10. is_international ─────────────────────────────────────
    intl = np.empty(n, dtype="int8")
    intl[legit_mask] = np.random.choice([0, 1], legit_count, p=[0.95, 0.05])
    intl[fraud_mask] = np.random.choice([0, 1], fraud_count, p=[0.60, 0.40])
    df["is_international"] = intl
    print("[10/13] is_international done")

    # ── 11. account_txn_count_30d ────────────────────────────────
    txn = np.empty(n, dtype="float32")
    txn[legit_mask] = np.random.exponential(8, legit_count)

    fraud_idx  = np.where(fraud_mask)[0]
    split      = fraud_count // 2
    txn[fraud_idx[:split]]  = np.random.exponential(2,  split)             # dormant
    txn[fraud_idx[split:]]  = np.random.exponential(40, fraud_count-split) # high velocity

    df["account_txn_count_30d"] = np.clip(txn, 1, 150).astype("int8")
    print("[11/13] account_txn_count_30d done")

    # ── 12. merchant_category ────────────────────────────────────
    mcat_opts = ["Retail", "Food", "Utilities", "Electronics", "Travel", "Gambling"]
    mcat = np.empty(n, dtype=object)
    mcat[legit_mask] = np.random.choice(mcat_opts, legit_count, p=[0.30, 0.25, 0.20, 0.15, 0.07, 0.03])
    mcat[fraud_mask] = np.random.choice(mcat_opts, fraud_count, p=[0.10, 0.05, 0.02, 0.38, 0.17, 0.28])
    df["merchant_category"] = pd.Categorical(mcat, categories=mcat_opts)
    print("[12/13] merchant_category done")

    # ── 13. customer_state ───────────────────────────────────────
    states = [
        "Maharashtra", "Delhi", "Karnataka", "Tamil Nadu",
        "Telangana", "Gujarat", "Rajasthan", "West Bengal",
        "Uttar Pradesh", "Punjab", "Haryana", "Kerala",
        "Madhya Pradesh", "Bihar", "Odisha", "Assam",
        "Jharkhand", "Uttarakhand", "Himachal Pradesh", "Goa"
    ]
    legit_p = np.ones(len(states)) / len(states)
    fraud_p = np.full(len(states), 0.40 / (len(states) - 4))
    fraud_p[0] = 0.18   # Maharashtra
    fraud_p[1] = 0.16   # Delhi
    fraud_p[2] = 0.14   # Karnataka
    fraud_p[3] = 0.12   # Tamil Nadu
    fraud_p    /= fraud_p.sum()

    st = np.empty(n, dtype=object)
    st[legit_mask] = np.random.choice(states, legit_count, p=legit_p)
    st[fraud_mask] = np.random.choice(states, fraud_count, p=fraud_p)
    df["customer_state"] = pd.Categorical(st, categories=states)
    print("[13/13] customer_state done")

    print(f"\nDone. Shape: {df.shape}")
    return df


# ── Quick validation ─────────────────────────────────────────────
def validate(df):
    fraud = df[df["isFraud"] == 1]
    legit = df[df["isFraud"] == 0]
    checks = [
        ("account_age_days avg",    f"{legit['account_age_days'].mean():.0f}d",   f"{fraud['account_age_days'].mean():.0f}d",   "fraud << legit"),
        ("is_new_account rate",     f"{legit['is_new_account'].mean()*100:.1f}%", f"{fraud['is_new_account'].mean()*100:.1f}%", "fraud >> legit"),
        ("is_international rate",   f"{legit['is_international'].mean()*100:.1f}%",f"{fraud['is_international'].mean()*100:.1f}%","fraud >> legit"),
        ("customer_age avg",        f"{legit['customer_age'].mean():.1f}",         f"{fraud['customer_age'].mean():.1f}",         "fraud > legit"),
        ("top merchant (fraud)",    "-", fraud['merchant_category'].value_counts().index[0], "Electronics or Gambling"),
        ("late_night fraud share",  f"{(legit['time_of_day']=='late_night').mean()*100:.1f}%",
                                    f"{(fraud['time_of_day']=='late_night').mean()*100:.1f}%","fraud >> legit"),
    ]
    print(f"\n{'─'*62}")
    print(f"{'Check':<28} {'Legit':>10} {'Fraud':>10}  {'Expected'}")
    print(f"{'─'*62}")
    for name, lv, fv, exp in checks:
        print(f"{name:<28} {lv:>10} {fv:>10}  {exp}")
    print(f"{'─'*62}\n")


# ── Run ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("Loading PaySim...")
    df = pd.read_csv("../data/PaySim_transaction_raw.csv")

    df_enriched = enrich_paysim(df)
    validate(df_enriched)

    df_enriched.to_parquet("paysim_enriched.parquet", index=False)
    print("Saved → paysim_enriched.parquet")

    print("\nFraud sample:")
    print(df_enriched[df_enriched["isFraud"] == 1].tail(5).to_string())

Loading PaySim...
Total : 6,362,620  |  Fraud : 8,213  |  Legit : 6,354,407

[1/13] transaction_hour done
[2/13] transaction_day_of_week done
[3/13] time_of_day done
[4/13] customer_age done
[5/13] customer_gender done
[6/13] account_age_days done
[7/13] is_new_account done
[8/13] device_type done
[9/13] channel done
[10/13] is_international done
[11/13] account_txn_count_30d done
[12/13] merchant_category done
[13/13] customer_state done

Done. Shape: (6362620, 24)

──────────────────────────────────────────────────────────────
Check                             Legit      Fraud  Expected
──────────────────────────────────────────────────────────────
account_age_days avg              1223d        46d  fraud << legit
is_new_account rate                8.3%      93.2%  fraud >> legit
is_international rate              5.0%      39.6%  fraud >> legit
customer_age avg                   37.8       51.3  fraud > legit
top merchant (fraud)                  - Electronics  Electronics or Gambli

In [2]:
df_enriched.to_csv("../data/paysim_transaction_enriched.csv", index = False)